# Chapter 7: Autopilot Models for Heading and Course Control

This introductory notebook bridges the theoretical foundations of surface vessel steering with practical, interactive implementations. We strip down complex marine dynamics into decoupled 1-DOF and 3-DOF tracking models optimized for real-time guidance, navigation, and control (GNC) pipelines.

---

## 7.1 Course Control Models

### What is a Transfer Function?
A **Transfer Function** is a mathematical representation that maps the input of a linear, time-invariant (LTI) system to its output in the frequency domain, using the Laplace variable $s$. 

* **Why is it necessary?** Instead of solving grueling differential equations in the time domain, a transfer function converts differentiation into simple multiplication ($s \cdot X(s)$). 
* **Its Purpose:** It allows GNC engineers to immediately analyze system stability (via poles and zeros), understand frequency responses (Bode plots), and design autopilot feedback loops without needing to track every internal state variable of the vessel hull.

### 7.1.1 State-Space Model for Course Control & 7.1.2 Course Angle Transfer Function
The relationship between heading angle ($\psi$), sideslip/drift ($\beta$), and the actual track over ground—the **Course Angle ($\chi$)**—is governed by:

$$\chi = \psi + \beta$$

Where $\beta = \arctan(v/u)$. If we assume a constant forward speed $U$ and linear steering characteristics, the transfer function mapping rudder input $\delta$ to course output $\chi$ scales the classic heading model by accounting for the lateral drift delay:

$$\frac{\chi}{\delta}(s) = \frac{K(1 + T_3 s)}{s(1 + T_1 s)(1 + T_2 s)}$$

Let's look at how the actual trajectory of the ship (Course) lags behind where the bow is pointing (Heading) due to this hydrodynamic sideslip.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import lsim, lti
import ipywidgets as widgets
from ipywidgets import VBox, HBox
%matplotlib inline

def demo_course_control(K, T1, T2, T3, rudder_angle, speed):
    t = np.linspace(0, 20, 1000)
    dt = t[1] - t[0]
    
    delta = np.zeros_like(t)
    delta[t >= 1.0] = np.radians(rudder_angle)
    
    # 2nd-Order Heading Model
    sys_heading = lti([K * T3, K], [T1 * T2, T1 + T2, 1])
    _, r, _ = lsim(sys_heading, U=delta, T=t)
    psi = np.cumsum(r) * dt
    
    # Approximate lateral velocity (v) causing sideslip (beta)
    v = -0.2 * speed * r 
    beta = np.arctan2(v, speed)
    chi = psi + beta  # Course angle
    
    plt.figure(figsize=(10, 4))
    plt.plot(t, np.degrees(psi), 'b-', label='Heading Angle ($\psi$)', linewidth=2)
    plt.plot(t, np.degrees(chi), 'g-.', label='Course Angle ($\chi = \psi + \beta$)', linewidth=2)
    plt.title("7.1.1 & 7.1.2 Course vs. Heading Tracking (Sideslip Lag)")
    plt.ylabel("Orientation [deg]")
    plt.xlabel("Time [s]")
    plt.legend()
    plt.grid(True)
    plt.show()

# Interactive Sliders for Section 7.1
widgets.interact(demo_course_control, 
                 K=(0.05, 0.5, 0.01), T1=(2.0, 20.0, 0.5), T2=(0.5, 5.0, 0.1), T3=(0.0, 4.0, 0.1),
                 rudder_angle=(-35.0, 35.0, 1.0), speed=(2.0, 15.0, 0.5))

## 7.2 Autopilot Models for Heading Control

### 7.2.1 Second-Order Nomoto Model
The second-order Nomoto model captures the critical high-frequency acceleration lag caused by the coupling between a vessel's sway (sideways slide) and yaw (turning) dynamics:

$$\frac{r}{\delta}(s) = \frac{K(1 + T_3 s)}{(1 + T_1 s)(1 + T_2 s)}$$

### 7.2.2 First-Order Nomoto Model
For basic autopilot design, tracking high-frequency sway oscillations is unnecessary. Nomoto demonstrated that by mathematically "lumping" the high-frequency lag poles into a single effective time constant ($T = T_1 + T_2 - T_3$), we can simplify the ship dynamics down to a clean first-order system:

$$T \dot{r} + r = K \delta \implies \frac{r}{\delta}(s) = \frac{K}{T s + 1}$$

### 💡 Autopilot Control Challenge: Overshoot vs. Over-damping
A ship possesses massive physical and added mass inertia, which is represented by a large time constant $T$. Once a massive hull starts rotating, its angular momentum is immense. 
* **The Problem:** If an autopilot acts too aggressively using purely Proportional (P) control, the ship will blast right past its target heading. If the controller overcorrects, it triggers violent, fuel-wasting **heading oscillations**.
* **The Solution:** Autopilots rely heavily on **Derivative (D) action**, which measures the *yaw rate* ($r$). The derivative term acts as an artificial damper. As the ship approaches the target heading, the D-term senses the high rotational velocity and preemptively commands counter-rudder to check the turn, settling the ship without overshoot or sluggish over-damping.

Let's evaluate how closely the first-order engineering approximation tracks the true second-order hull dynamics.

In [ ]:
def demo_nomoto_order(K, T1, T2, T3, rudder_angle):
    t = np.linspace(0, 20, 1000)
    delta = np.zeros_like(t)
    delta[t >= 1.0] = np.radians(rudder_angle)
    
    # Second-Order Nomoto Model
    sys_2nd = lti([K * T3, K], [T1 * T2, T1 + T2, 1])
    _, r_2nd, _ = lsim(sys_2nd, U=delta, T=t)
    
    # First-Order Equivalent
    T_eff = max(0.1, T1 + T2 - T3)
    sys_1st = lti([K], [T_eff, 1])
    _, r_1st, _ = lsim(sys_1st, U=delta, T=t)
    
    plt.figure(figsize=(10, 4))
    plt.plot(t, np.degrees(r_1st), 'r--', label='1st-Order Nomoto Approximation', linewidth=2)
    plt.plot(t, np.degrees(r_2nd), 'b-', label='2nd-Order Nomoto Model', linewidth=2)
    plt.title("7.2.1 & 7.2.2 Heading Autopilot Model Comparison")
    plt.ylabel("Yaw Rate [deg/s]")
    plt.xlabel("Time [s]")
    plt.legend()
    plt.grid(True)
    plt.show()

widgets.interact(demo_nomoto_order, 
                 K=(0.05, 0.5, 0.01), T1=(2.0, 20.0, 0.5), T2=(0.5, 5.0, 0.1), T3=(0.0, 4.0, 0.1),
                 rudder_angle=(-35.0, 35.0, 1.0))

### 7.2.3 Norrbin Nonlinear Extension
When a vessel undergoes large, radical maneuvers, or if the hull design is directionally unstable, the linear Nomoto models fail. 

Norrbin extended the framework by swapping out the linear damping term for a cubic non-linear steering characteristic polynomial, $n(r)$:

$$T \dot{r} + n(r) = K \delta \quad \text{where } n(r) = \alpha_1 r + \alpha_3 r^3$$

### 💡 Autopilot Control Challenge: Overcoming the Destabilizing Munk Moment
As a slender hull travels forward and begins to drift or turn, it experiences a powerful hydrodynamic destabilizing torque known as the **Munk moment**. The water pressure builds at the bow and drops at the stern, actively trying to twist the ship *further* into the turn.

In Norrbin's model, this maps directly to the linear coefficient $\alpha_1$:
* **Directionally Stable Hull ($\alpha_1 > 0$):** The hull naturally resists drifting and will self-straighten.
* **Directionally Unstable Hull ($\alpha_1 < 0$):** The Munk moment completely overpowers the hull's natural linear damping. Without active control, the ship cannot travel in a straight line; it will immediately break into an uncontrolled port or starboard spiral.

* **The Autopilot Solution:** The autopilot's Proportional gain ($K_p$) acts as an artificial, positive spring. By modulating the rudder, it generates an active counter-torque that overpowers the negative spring of the Munk moment, forcing an unstable hull to behave safely and predictably. Meanwhile, the cubic drag term ($\alpha_3$) acts as a high-velocity anchor, nonlinearly draining energy out of the turn and capping the maximum velocity the ship can reach.

In [ ]:
def demo_norrbin(K, T1, T2, T3, rudder_angle, alpha3):
    t = np.linspace(0, 20, 1000)
    dt = t[1] - t[0]
    delta = np.zeros_like(t)
    delta[t >= 1.0] = np.radians(rudder_angle)
    
    T_eff = max(0.1, T1 + T2 - T3)
    
    # Linear baseline
    sys_lin = lti([K], [T_eff, 1])
    _, r_lin, _ = lsim(sys_lin, U=delta, T=t)
    
    # Nonlinear Norrbin via Euler integration
    r_nl = np.zeros_like(t)
    for i in range(1, len(t)):
        r_dot = (K * delta[i-1] - r_nl[i-1] - alpha3 * (r_nl[i-1]**3)) / T_eff
        r_nl[i] = r_nl[i-1] + r_dot * dt
        
    plt.figure(figsize=(10, 4))
    plt.plot(t, np.degrees(r_lin), 'b--', label='Linear Nomoto Baseline', alpha=0.6)
    plt.plot(t, np.degrees(r_nl), 'm-', label=f'Norrbin Nonlinear Model ($\\alpha_3$ = {alpha3})', linewidth=2)
    plt.title("7.2.3 Norrbin Nonlinear Drag Saturation")
    plt.ylabel("Yaw Rate [deg/s]")
    plt.xlabel("Time [s]")
    plt.legend()
    plt.grid(True)
    plt.show()

widgets.interact(demo_norrbin, 
                 K=(0.05, 0.5, 0.01), T1=(2.0, 20.0, 0.5), T2=(0.5, 5.0, 0.1), T3=(0.0, 4.0, 0.1),
                 rudder_angle=(-35.0, 35.0, 1.0), alpha3=(0.0, 10.0, 0.5))

### 7.2.4 The Pivot Point
Kinematically, when a surface ship stabilizes into a turning circle, there is a singular longitudinal location along the hull's centerline where the local transverse velocity vector ($v$) completely vanishes. 

This location is called the **Pivot Point ($x_p$)**:

$$x_p = \frac{v}{r}$$

To an observer standing on the deck at $x_p$, the ship does not feel like it is sliding sideways at all; it feels like the entire vessel is pure-pivoting around that exact point. During forward travel, the pivot point migrates well forward of the Center of Gravity (CG) toward the bow.

In [ ]:
def demo_pivot_point(K, T1, T2, T3, rudder_angle, speed):
    t = np.linspace(0, 20, 1000)
    delta = np.zeros_like(t)
    delta[t >= 1.0] = np.radians(rudder_angle)
    
    sys_2nd = lti([K * T3, K], [T1 * T2, T1 + T2, 1])
    _, r, _ = lsim(sys_2nd, U=delta, T=t)
    
    v = -0.2 * speed * r
    
    x_p = np.zeros_like(t)
    for i in range(len(t)):
        if abs(r[i]) > 1e-4:
            x_p[i] = v[i] / r[i]
            
    plt.figure(figsize=(10, 4))
    # Filter transient startup to avoid infinite zero division spikes
    plt.plot(t[t > 1.5], x_p[t > 1.5], 'k-', label='Pivot Point Position ($x_p$)', linewidth=2)
    plt.axhline(0, color='r', linestyle=':', label='Vessel Center of Gravity (CG)')
    plt.title("7.2.4 Kinematic Pivot Point Shift Location Forward of CG")
    plt.ylabel("Distance Forward of CG [m]")
    plt.xlabel("Time [s]")
    plt.legend()
    plt.grid(True)
    plt.show()

widgets.interact(demo_pivot_point, 
                 K=(0.05, 0.5, 0.01), T1=(2.0, 20.0, 0.5), T2=(0.5, 5.0, 0.1), T3=(0.0, 4.0, 0.1),
                 rudder_angle=(-35.0, 35.0, 1.0), speed=(2.0, 15.0, 0.5))

## Summary: Combining the Pieces Together

We have built up the surface autopilot story piece by piece:
1. **Nomoto Models (7.2.1, 7.2.2)** capture how command outputs transform into physical hull rotation rates ($r$) while managing massive inertial momentum.
2. **Norrbin Extensions (7.2.3)** step in to counteract course instability from the Munk moment and bound peak yaw rates nonlinearly.
3. **The Pivot Point (7.2.4)** defines the kinematic center of rotation, revealing the structural relationship between rotation ($r$) and translation ($v$).
4. **Course Models (7.1.1, 7.1.2)** overlay the resulting spatial drift ($\beta$), tracking how the overall trajectory slices through coordinates over ground.

### 💡 Real-World Note: Limit Cycles and Wave Filtering
In open seas, high-frequency waves will constantly rock the hull back and forth. If the autopilot tries to correct for every single wave hit, the rudder will rapidly cycle left and right. This "rudder chaffing" doesn't actually steer the massive ship—it just destroys the steering hydraulics. 

To prevent this, real autopilot architectures implement **low-pass wave filters** (like a Kalman filter) and **nonlinear deadbands** to ignore high-frequency oscillations, ensuring the rudder only fights true, low-frequency course drift.

Run the final grand dashboard cell below to tune all these interconnected parameters simultaneously!

In [ ]:
# --- FINAL COMBINED MATRIX DASHBOARD ---
def simulate_grand_dashboard(K, T1, T2, T3, rudder_angle, alpha3, speed):
    t = np.linspace(0, 20, 1000)
    dt = t[1] - t[0]
    delta = np.zeros_like(t)
    delta[t >= 1.0] = np.radians(rudder_angle)
    T_eff = max(0.1, T1 + T2 - T3)
    
    sys_1st = lti([K], [T_eff, 1])
    _, r_1st, _ = lsim(sys_1st, U=delta, T=t)
    sys_2nd = lti([K * T3, K], [T1 * T2, T1 + T2, 1])
    _, r_2nd, _ = lsim(sys_2nd, U=delta, T=t)
    
    psi_2nd = np.cumsum(r_2nd) * dt
    v_2nd = -0.2 * speed * r_2nd 
    beta = np.arctan2(v_2nd, speed)
    chi_2nd = psi_2nd + beta
    
    r_nl = np.zeros_like(t)
    for i in range(1, len(t)):
        r_dot = (K * delta[i-1] - r_nl[i-1] - alpha3 * (r_nl[i-1]**3)) / T_eff
        r_nl[i] = r_nl[i-1] + r_dot * dt
        
    x_p = np.zeros_like(t)
    for i in range(len(t)):
        if abs(r_2nd[i]) > 1e-4: x_p[i] = v_2nd[i] / r_2nd[i]

    fig, axs = plt.subplots(2, 2, figsize=(15, 8))
    axs[0, 0].plot(t, np.degrees(r_1st), 'r--', t, np.degrees(r_2nd), 'b-')
    axs[0, 0].set_title("7.2.1 & 7.2.2 Yaw Rates"); axs[0, 0].set_ylabel("[deg/s]")
    
    axs[0, 1].plot(t, np.degrees(psi_2nd), 'b-', t, np.degrees(chi_2nd), 'g-.')
    axs[0, 1].set_title("7.1.1 & 7.1.2 Course vs Heading"); axs[0, 1].set_ylabel("[deg]")
    
    axs[1, 0].plot(t, np.degrees(r_2nd), 'b--', t, np.degrees(r_nl), 'm-')
    axs[1, 0].set_title("7.2.3 Norrbin Damping Saturation"); axs[1, 0].set_ylabel("[deg/s]")
    
    axs[1, 1].plot(t[t>1.5], x_p[t>1.5], 'k-'); axs[1, 1].axhline(0, color='r', linestyle=':')
    axs[1, 1].set_title("7.2.4 Pivot Point Position"); axs[1, 1].set_ylabel("[m forward]")
    
    for ax in axs.flat: ax.grid(True); ax.set_xlabel("Time [s]")
    plt.tight_layout()
    plt.show()

style = {'description_width': 'initial'}
k_w = widgets.FloatSlider(value=0.15, min=0.05, max=0.5, step=0.01, description='Gain K:', style=style)
t1_w = widgets.FloatSlider(value=10.0, min=2.0, max=20.0, step=0.5, description='T1:', style=style)
t2_w = widgets.FloatSlider(value=2.0, min=0.5, max=5.0, step=0.1, description='T2:', style=style)
t3_w = widgets.FloatSlider(value=1.0, min=0.0, max=4.0, step=0.1, description='T3:', style=style)
rud_w = widgets.FloatSlider(value=15.0, min=-35.0, max=35.0, step=1.0, description='Rudder (deg):', style=style)
a3_w = widgets.FloatSlider(value=2.5, min=0.0, max=10.0, step=0.5, description='Norrbin a3:', style=style)
spd_w = widgets.FloatSlider(value=8.0, min=2.0, max=15.0, step=0.5, description='Speed U (m/s):', style=style)

ui = HBox([VBox([k_w, t1_w, t2_w, t3_w]), VBox([rud_w, a3_w, spd_w])])
out = widgets.interactive_output(simulate_grand_dashboard, {'K': k_w, 'T1': t1_w, 'T2': t2_w, 'T3': t3_w, 'rudder_angle': rud_w, 'alpha3': a3_w, 'speed': spd_w})
display(ui, out)

## Chapter 7 Blueprint: The Autopilot Architecture Summary

Congratulations! You have just stepped through the classical building blocks of marine steering control. When architecting a real-world Guidance, Navigation, and Control (GNC) firmware pipeline, these individual math sections form a layered stack:

┌─────────────────────────────────────────────────────────────────┐
│              OUTER LOOP: Course Control (7.1)                  │
│  Tracks coordinates across the Earth (GPS/GNSS). Calculates    │
│  cross-track error and compensates for sideslip drift (β).      │
└────────────────────────────────┬────────────────────────────────┘
│
▼ Outputs Commanded Heading (ψ_d)
┌─────────────────────────────────────────────────────────────────┐
│             INNER LOOP: Heading Autopilot (7.2)                 │
│  Reads high-frequency gyro data. Uses Nomoto dynamics (1st/2nd   │
│  order) to compute the required physical rudder torque.         │
└────────────────────────────────┬────────────────────────────────┘
│
▼ Outputs Rudder Command (δ)
┌─────────────────────────────────────────────────────────────────┐
│             ACTUATOR & HULL PLANT DYNAMICS                      │
│  - Actuator Rate Limits damp transient spikes.                  │
│  - Norrbin Damping (7.2.3) bounds peak yaw rates nonlinearly.   │
│  - Kinematic Pivot Point (7.2.4) dictates the turn rotation.    │
└─────────────────────────────────────────────────────────────────┘

### Key Takeaways for Your Design Toolbox:

1. **Heading vs. Course:** Keeping the bow pointed East ($90^\circ$ heading) is **not** the same as traveling East over ground. Wind and currents slide the hull sideways. Your control loops must actively calculate sideslip ($\beta$) to maintain a true spatial track.
2. **The Power of Order Reduction:** While a ship physically sways and yaws simultaneously (**2nd-Order Nomoto**), we can safely "compress" that high-frequency lag into a single effective time constant ($T$). This **1st-Order Nomoto** model gives us a highly clean, reliable plant for tuning standard PID heading controllers.
3. **Linearity is a Luxury:** Small rudder tweaks on a cargo ship follow elegant linear rules. Hard maneuvers or course-unstable hulls do not. The **Norrbin Model** acts as our guardrail, capturing how water drag exponentially limits a vessel's turning performance at high velocities.
4. **The Ship is a Pivot Geometry:** A ship does not turn like a car. It actively skids. Recognizing that the **Pivot Point** moves forward toward the bow during transit is essential when placing sensors (IMUs) or calculating hull clearances in tight channels.